In [ ]:
import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import (
    build_graph_model,
    cell_type_gradients,
    feature_gradients,
    load_graph_checkpoint,
    predict_single_graph,
)
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

### Load dataset

In [ ]:
# create the dataset
dataset = AD_Mouse(
    AD_adata_path=args.AD_adata_path,
    Wild_type_adata_path=args.Wild_type_adata_path,
    n_top_genes=args.n_top_genes,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Load model

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=args.use_cell_type).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_ct_STARmap_PLUS.pth', device=device)


# Inference 

In [ ]:
graph = dataset.testing[0]
pred_value, _, test_mask = predict_single_graph(model, graph, device=device, apply_sigmoid=True)
outputs_list = (pred_value[:, 0] > 0.5).numpy()
selected_mask = test_mask & torch.from_numpy(outputs_list)
print(f"Selected {int(selected_mask.sum())} A?-positive nodes for niche attribution.")


In [ ]:
gradients, outputs, attribution_mask = feature_gradients(
    model,
    graph,
    target_index=0,
    split='test',
    device=device,
    apply_sigmoid=True,
    selection_mask=selected_mask,
)

attributions = np.abs(gradients.numpy())


In [ ]:
adata = sc.read_h5ad('/home/wzk/ST_code/NicheTrans/2024_NicheTrans_upload_data/2023_nn_AD_mouse/AD_model_adata_protein/13months-disease-replicate_2_random.h5ad')
adata.obs['pred'] = outputs_list

# C1qa  Cst7  Cst3  Tpt1
# Zhx1  Xbp1  Nrgn  Trf
# change the name of genes for different visualization
target = 'Cst7'

graph_node_names = [node_id.split('/', 1)[1] for node_id in graph.node_ids]
node_name_to_index = {node_name: idx for idx, node_name in enumerate(graph_node_names)}
selected_source_nodes = [graph_node_names[idx] for idx, keep in enumerate(outputs_list) if keep]
abeta_positive_index = np.array(selected_source_nodes)

gene_index = int(np.flatnonzero(dataset.source_panel == target)[0])
edge_index = graph.edge_index.detach().cpu().numpy()
graph_neighbors = {node_name: [] for node_name in graph_node_names}
selected_attributions = []

for source_name in abeta_positive_index:
    source_idx = node_name_to_index[source_name]
    neighbor_indices = edge_index[0, edge_index[1] == source_idx].tolist()
    neighbor_names = [graph_node_names[idx] for idx in neighbor_indices]
    neighbor_scores = attributions[neighbor_indices, gene_index]
    graph_neighbors[source_name] = neighbor_names
    selected_attributions.append(neighbor_scores)

dataset.graph = graph_neighbors


In [ ]:
adata = adata[:, dataset.rna_mask]
adata.var_names = dataset.source_panel

In [ ]:
import matplotlib.cm as cm

cmap = cm.get_cmap('Reds')
max_ = max((scores.max() for scores in selected_attributions if len(scores) > 0), default=1.0)
selected_attributions_ = [scores / max_ if len(scores) > 0 else scores for scores in selected_attributions]
color = [cmap(scores) for scores in selected_attributions_]


In [ ]:
import matplotlib.patches as patches

In [ ]:
expression_list = []
attribution_list = []

fig, ax = plt.subplots(1, figsize=(4, 4), dpi=300)

ax.set_axis_off()
ax.set_frame_on(False)

sc.pl.embedding(adata, basis='spatial', color=target, ax=ax, show=False, s=5)

for i, source_index in enumerate(abeta_positive_index):
    neighbors = dataset.graph[source_index]

    for j, target_index in enumerate(neighbors):
        x1, y1 = adata[source_index].obsm['spatial'][0]
        x2, y2 = adata[target_index].obsm['spatial'][0]

        expression_list.append(adata[target_index, target].X.item())
        attribution_list.append(selected_attributions_[i][j].item())

        if selected_attributions_[i][j] > 0.2:
            ax.annotate('',
                        xy=(x1, y1),
                        xytext=(x2, y2),
                        arrowprops=dict(
                            arrowstyle='->',
                            color=color[i][j],
                            lw=.5,
                            shrinkA=0, shrinkB=0,
                            mutation_scale=3,
                        )
                    )

ax.scatter(adata.obsm['spatial'][outputs_list, 0], adata.obsm['spatial'][outputs_list, 1], color='black', s=1)

rect = patches.Rectangle((3200, 1200), 1100, 1100, linewidth=1., edgecolor='red', facecolor='none')
ax.add_patch(rect)

rect = patches.Rectangle((4900, 4600), 1100, 1100, linewidth=1., edgecolor='orange', facecolor='none')
ax.add_patch(rect)


In [ ]:
from scipy.stats import pearsonr, spearmanr
pcc, p_value_pcc = pearsonr(attribution_list, expression_list)
spcc, p_value_spcc = spearmanr(attribution_list, expression_list)

print(f"Pearson correlation coefficient (PCC): {pcc:.4f}, p-value: {p_value_pcc:.4f}")
print(f"Spearman correlation coefficient (SPCC): {spcc:.4f}, p-value: {p_value_spcc:.4f}")

fig, ax = plt.subplots(figsize=(4,4), dpi=300)

ax.scatter(attribution_list, expression_list, color='blue', marker='o', s=2)  # 你也可以改颜色或点的形状

# plt.title("散点图示例", fontsize=14)
ax.set_xlabel("Attribution score", fontsize=16)
ax.set_ylabel("Gene expression value", fontsize=16)
ax.set_xlim(0, 1)
ax.tick_params(axis='both', labelsize=16)

plt.show()

This part is for visualizing the interactions within niches. 

In [ ]:
# import matplotlib.patches as mpatches

# save_dir = f'./niche_level_{target}_attribution/'
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)

# for index, source_index in enumerate(abeta_positive_index):

#     fig, ax1 = plt.subplots(1, figsize=(1.5, 1.5), dpi=300)
    
#     source_index = abeta_positive_index[index]
#     neighbors = dataset.graph[source_index]
#     sc.pl.embedding(adata[[source_index] + neighbors], basis='spatial', color=target, ax=ax1, show=False, s=25, colorbar_loc=None, vmax=adata[:, target].X.max())

#     # labels = adata[[source_index] + neighbors].obs[target].values

#     for j, target_index in enumerate(neighbors):

#         x1, y1 = adata[source_index].obsm['spatial'][0]
#         x2, y2 = adata[target_index].obsm['spatial'][0]

#         ax1.annotate('',                            
#                     xy=(x1, y1), 
#                     xytext=(x2, y2), 
#                     arrowprops=dict(
#                         arrowstyle='->', 
#                         color=color[index ,j], 
#                         lw=.8, 
#                         shrinkA=0, shrinkB=4, 
#                         mutation_scale=4, 
#                         # alpha=0.1
#                     ),
#                     zorder=0
#                 )

#     ax1.scatter(x1, y1, facecolors='none', edgecolors='grey', s=50, linewidths=0.5)
#     ax1.set_title(source_index)

#     ax1.set_axis_off()
#     ax1.set_frame_on(False)
#     plt.title('')
#     plt.savefig(save_dir + f"{source_index}.png", dpi=300, bbox_inches='tight')
#     plt.close()

In [ ]:
fig, ax = plt.subplots(1, figsize=(3, 3), dpi=300)
sc.pl.embedding(adata, basis='spatial', color=target, ax=ax, show=False, s=15, legend_loc=None)

ax.set_axis_off()
ax.set_frame_on(False)

for i, source_index in enumerate(abeta_positive_index):
    neighbors = dataset.graph[source_index]

    for j, target_index in enumerate(neighbors):
        x1, y1 = adata[source_index].obsm['spatial'][0]
        x2, y2 = adata[target_index].obsm['spatial'][0]

        if selected_attributions_[i][j] > 0.2:
            ax.annotate('',
                        xy=(x1, y1),
                        xytext=(x2, y2),
                        arrowprops=dict(
                            arrowstyle='->',
                            color=color[i][j],
                            lw=0.5,
                            shrinkA=1, shrinkB=1,
                            mutation_scale=4,
                            connectionstyle=f"arc3,rad={np.random.uniform(-0.4, 0.4)}"
                        )
                    )

ax.scatter(adata.obsm['spatial'][outputs_list, 0], adata.obsm['spatial'][outputs_list, 1], facecolors='none', edgecolors='black', s=20, linewidths=0.1)
ax.set_xlim(3200, 4300)
ax.set_ylim(1200, 2300)
ax.set_title('')

if ax.get_figure().axes[-1] != ax:
    ax.get_figure().axes[-1].remove()

plt.show()


In [ ]:
fig, ax = plt.subplots(1, figsize=(3, 3), dpi=300)
sc.pl.embedding(adata, basis='spatial', color=target, ax=ax, show=False, s=15, legend_loc=None)

ax.set_axis_off()
ax.set_frame_on(False)

for i, source_index in enumerate(abeta_positive_index):
    neighbors = dataset.graph[source_index]

    for j, target_index in enumerate(neighbors):
        x1, y1 = adata[source_index].obsm['spatial'][0]
        x2, y2 = adata[target_index].obsm['spatial'][0]

        if selected_attributions_[i][j] > 0.2:
            ax.annotate('',
                        xy=(x1, y1),
                        xytext=(x2, y2),
                        arrowprops=dict(
                            arrowstyle='->',
                            color=color[i][j],
                            lw=0.5,
                            shrinkA=1, shrinkB=1,
                            mutation_scale=4,
                            connectionstyle=f"arc3,rad={np.random.uniform(-0.4, 0.4)}"
                        )
                    )

ax.scatter(adata.obsm['spatial'][outputs_list, 0], adata.obsm['spatial'][outputs_list, 1], facecolors='none', edgecolors='black', s=40, linewidths=0.1)
ax.set_xlim(4900, 6000)
ax.set_ylim(4600, 5700)
ax.set_title('')

if ax.get_figure().axes[-1] != ax:
    ax.get_figure().axes[-1].remove()

plt.show()
